| Point                          | Approach 1: Continue Same Stage 1 LoRA Adapter                                         | Approach 2: Merge Stage 1, Then Add New LoRA Adapter                                                      |
| ------------------------------ | -------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- |
| Flow                           | Base model + Stage 1 LoRA adapter → continue training same adapter on instruction data | Base model + Stage 1 LoRA adapter → merge → load merged model → add new LoRA adapter for instruction data |
| Main idea                      | The same adapter learns both domain language and instruction-following behavior        | Stage 1 knowledge becomes part of the merged model, then a new adapter learns instruction behavior        |
| Simplicity                     | Easier for students to understand                                                      | More complex because merge step is involved                                                               |
| Merge required before Stage 2? | No                                                                                     | Yes                                                                                                       |
| Risk/complexity                | Lower complexity                                                                       | Higher complexity, especially if Stage 1 model was loaded in 4-bit/QLoRA mode                             |
| Output size                    | Small final LoRA adapter                                                               | Large merged Stage 1 model + small Stage 2 adapter                                                        |
| Hugging Face upload            | Easy, only adapter can be pushed                                                       | Heavier because merged model is large                                                                     |
| Best for course/demo           | Best choice                                                                            | Use only if you already merged Stage 1                                                                    |
| Best for production            | Good for experimentation and adapter-based deployment                                  | Useful when you want Stage 1 knowledge permanently inside the base model                                  |
| Recommended for your notebook? | **Yes, recommended**                                                                   | Only if Stage 1 adapter is already merged                                                                 |


# **Following Approach 2**
```
Stage 1 ───> BASE model + LoRA Adapter 1 <─── Non-inst data
                   │                 │
                   └───────┬─────────┘
                         merge
                           │
                           ▼
Stage 2 ───> BASE MODEL + LoRA Adapter 2 <─── Instruction data
                   │                 │
                   └───────┬─────────┘
                         merge
                           │
                           ▼
Stage 3 ───> BASE model + LoRA Adapter 3 <─── Preference data
                   │                 │
                   └───────┬─────────┘
                         merge

```

# **End to End Pipeline of all three stages**

Non-instruction FT on raw data  
↓  
Save the model  
↓  
Load the model  
↓  
Perform instruction FT on same model  
↓  
Save the model  
↓  
Load the model again  
↓  
Perform preference tuning on top of it

## **Stage 1: Non-instruction Causal LLM Fine-tuning or Domain-Adaptive Continued Pretraining**

### **Pipeline for first stage**

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

### Goal

In this stage1, we are performing continued pretraining / non-instruction fine-tuning on raw pharma PDF text.

The model is given raw domain text such as:
> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to predict the next token from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is not explicitly taught:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

* These all things are taught in Supervised FineTuning




In [1]:
# ============================================================
# 1. Install required libraries
# ============================================================
# PyMuPDF: PDF text extraction
# datasets: Hugging Face dataset creation
# transformers/accelerate: model, tokenizer, Trainer
# peft: LoRA/QLoRA adapters
# bitsandbytes: 4-bit/8-bit quantized loading

!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes torchao

In [2]:

# ============================================================
# 2. Imports
# ============================================================

import os
import re
import gc
import math
import json
import random
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import fitz
import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [54]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keeping all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge (1).pdf"

    # Base causal language model that we will fine-tune on pharma-domain text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/content/pharma_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "/content/pharma_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 10.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

In [55]:
config = Config()

In [56]:
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge (1).pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=10.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [7]:
import json
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge (1).pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 8.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [8]:
config.output_dir

'/content/pharma_tinyllama_lora_output'

In [9]:
config.processed_data_dir

'/content/pharma_processed_data'

In [10]:
# creating output directory code
import os
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [11]:
# ============================================================
# Hugging Face repo names
# ============================================================

HF_USERNAME = "piyushh45"

BASE_MODEL_NAME = config.model_name

# Stage 1: Non-instruction LoRA adapter
HF_REPO_NON_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-lora-adapter"

# Stage 1 merged model
HF_REPO_NON_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-merged"

# Stage 2: Instruction LoRA adapter
HF_REPO_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-instruction-lora-adapter"

# Stage 2 merged model
HF_REPO_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-instruction-merged"

# Stage 3: DPO preference LoRA adapter
HF_REPO_DPO_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-dpo-lora-adapter"

# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"{HF_USERNAME}/pharma-tinyllama-dpo-merged"

print(HF_REPO_NON_INSTRUCTION_ADAPTER)
print(HF_REPO_INSTRUCTION_ADAPTER)
print(HF_REPO_DPO_ADAPTER)

piyushh45/pharma-tinyllama-non-instruction-lora-adapter
piyushh45/pharma-tinyllama-instruction-lora-adapter
piyushh45/pharma-tinyllama-dpo-lora-adapter


In [12]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge (1).pdf


In [13]:
# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages


In [14]:
config.pdf_path

'/content/Metformin-Lipid-Therapy-Knowledge (1).pdf'

In [15]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [16]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [17]:
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

In [18]:
# ============================================================
# 6. Text cleaning utilities
# ============================================================

In [19]:
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [20]:
cleaned_pages = []
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

In [21]:
print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 6


In [22]:
print(cleaned_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogenous cholesterol synthesis

In [23]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# This step converts cleaned page-level text into paragraph-level records.

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [24]:
paragraph_records = split_into_paragraph_records(cleaned_pages)
print("Total paragraph records:", len(paragraph_records))

Total paragraph records: 9


In [25]:
# Printing cleaned paragraphs
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

Page: 1 | Paragraph: 1 | Characters: 575
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Page: 1 | Paragraph: 2 | Characters: 598
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin

In [26]:
# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /content/pharma_processed_data/pharma_paragraph_process.jsonl


In [27]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================
from datasets import Dataset
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )
text_dataset = Dataset.from_list(paragraph_records)


In [28]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [29]:
print(text_dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


In [30]:
# ============================================================
# 10. Train/eval split
# ============================================================

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

from datasets import DatasetDict
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


In [31]:
# ============================================================
# 11. Load tokenizer in RAM
# ============================================================
# tokenizer converts text into token_ids
# use snapshot to load tokenizer locally

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [32]:
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


In [33]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"]) # takes only text column and convert it into token_ids

In [34]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names, #After tokenization, the original dataset columns are removed. Only tokenized columns such as `input_ids` and `attention_mask` remain.
    desc="Tokenizing text corpus", #Just a progress bar description
)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

In [35]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2
    })
})

### Purpose of create_training_blocks function

This function converts tokenized text into **fixed-size training blocks**.
First, it joins all token IDs into one long sequence.  
Then, it cuts that long sequence into equal blocks of `config.block_size` tokens.
For causal language modeling, the labels are copied from `input_ids` because the model learns to predict the next token.

#### Example

Suppose we have these tokenized inputs:
``` text
Input token lists:
[10, 20, 30]
[40, 50]
[60, 70, 80, 90]
After joining all token lists together:
[10, 20, 30, 40, 50, 60, 70, 80, 90]

If:
block_size = 4
Then the final training blocks become:
Block 1 = [10, 20, 30, 40]
Block 2 = [50, 60, 70, 80]
The remaining token is:
[90]
This token is dropped because it cannot form a complete block of 4 tokens.



In [36]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [37]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/7 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/2 [00:00<?, ? examples/s]

In [38]:
final_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 0
    })
})

In [39]:
sample = final_dataset["train"][0]

In [40]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals,

In [41]:
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [42]:
# Clear memory before loading the model.
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()


In [43]:
# ============================================================
# 13. Load base model with QLORA Config
# ============================================================
from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


In [44]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.

from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [45]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [46]:
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


#### need for datacollator

Even after we have already packed the tokens into fixed-size blocks, the Data Collator is still needed because those blocks must be grouped into batches during training, and the collator dynamically pads sequences within each batch to the same length, correctly prepares the labels for next-token prediction, and creates proper attention masks so the model ignores padding tokens — without it, the training loop would fail when trying to stack examples of slightly different effective lengths or when forming the final tensors that the model expects.

In [47]:
# ============================================================
# 15. Data collator
# ============================================================
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [58]:
# ============================================================
# 16. Training arguments
# ============================================================
from transformers import TrainingArguments

In [59]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [60]:
training_args = TrainingArguments(**training_kwargs)
training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_star

In [61]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


In [62]:
import warnings
warnings.filterwarnings("ignore")

The Trainer is responsible for reading the dataset, computing the loss, performing backpropagation, and updating the model weights during training. When using LoRA, it only updates the small set of LoRA adapter weights instead of the full model. Importantly, the Trainer itself does not become the LoRA adapter — it is simply the training engine. After training finishes, the model consists of the original frozen base model plus the newly trained LoRA adapter that has been added on top of it.

In [63]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()
print("Training completed.")

Step,Training Loss
1,1.615512
2,1.613985
3,1.577430
4,1.502645
5,1.389241
6,1.244322
7,1.080736
8,0.958467
9,0.866295
10,0.804130


Training completed.


In [64]:
for log in trainer.state.log_history:
    print(log)

{'loss': 1.6155116558074951, 'grad_norm': 0.6772618293762207, 'learning_rate': 0.0, 'epoch': 1.0, 'step': 1}
{'loss': 1.6139850616455078, 'grad_norm': 0.7028564214706421, 'learning_rate': 4e-05, 'epoch': 2.0, 'step': 2}
{'loss': 1.5774304866790771, 'grad_norm': 0.7393331527709961, 'learning_rate': 8e-05, 'epoch': 3.0, 'step': 3}
{'loss': 1.5026450157165527, 'grad_norm': 0.8286087512969971, 'learning_rate': 0.00012, 'epoch': 4.0, 'step': 4}
{'loss': 1.3892408609390259, 'grad_norm': 0.9656257629394531, 'learning_rate': 0.00016, 'epoch': 5.0, 'step': 5}
{'loss': 1.2443217039108276, 'grad_norm': 0.9141824245452881, 'learning_rate': 0.0002, 'epoch': 6.0, 'step': 6}
{'loss': 1.0807361602783203, 'grad_norm': 0.9335317015647888, 'learning_rate': 0.00016, 'epoch': 7.0, 'step': 7}
{'loss': 0.9584668874740601, 'grad_norm': 1.0562325716018677, 'learning_rate': 0.00012, 'epoch': 8.0, 'step': 8}
{'loss': 0.866294801235199, 'grad_norm': 0.9231087565422058, 'learning_rate': 8e-05, 'epoch': 9.0, 'step'

During training, the tokenizer is first loaded from the base model (TinyLlama) and used to convert the raw text dataset into numerical token IDs. Once the dataset is tokenized, it only stores these numbers and no longer contains the mapping between words and token IDs. That mapping lives inside the tokenizer itself. Therefore, after training, the same tokenizer is saved along with the LoRA adapter so that it can later convert new text into the correct token IDs during inference. Although it is possible to reload the tokenizer directly from TinyLlama, saving it with the adapter is preferred because it guarantees an exact match with the version used during training, keeps the adapter self-contained, and prevents future mismatches if the original tokenizer is updated.

In [65]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

('/content/pharma_tinyllama_lora_adapter/tokenizer_config.json',
 '/content/pharma_tinyllama_lora_adapter/tokenizer.json')

In [66]:
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['adapter_config.json', 'README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [67]:
HF_REPO_NON_INSTRUCTION_ADAPTER

'piyushh45/pharma-tinyllama-non-instruction-lora-adapter'

In [68]:
from huggingface_hub import login
login()

In [69]:
# ============================================================
# Push Stage 1 non-instruction LoRA adapter to Hugging Face
# ============================================================

model.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=True
)


tokenizer.push_to_hub(
    HF_REPO_NON_INSTRUCTION_ADAPTER,
    private=True
)

print("Stage 1 non-instruction LoRA adapter pushed to:")
print(HF_REPO_NON_INSTRUCTION_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 1 non-instruction LoRA adapter pushed to:
piyushh45/pharma-tinyllama-non-instruction-lora-adapter


In [70]:
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [71]:
# ============================================================
# 21. Reload tokenizer for inferencing
# ============================================================
from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [72]:
# ============================================================
#  load base model
# ============================================================
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [73]:
# ============================================================
#  merges trained lora adapter on base model
# ============================================================
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)


In [74]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [75]:
print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


In [76]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations, not chat-style questions.


def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [77]:
# ============================================================
# 23. Test text continuation
# ============================================================

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]


In [78]:
# ============================================================
# 23. Test text continuation
# ============================================================

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents used in type 2 diabetes mellitus. It works by inhibiting insulin secretion and increasing insulin sensitivity in the liver, muscle, and peripheral tissues. In people with type 2 diabetes, Metformin improves glycemic control, reduces cardiovascular risk, and may reduce microvascular complications such as neuropathy and retinopathy. Metformin has also been shown to improve weight loss and metabolic health in obese patients. Common side effects

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe improves cholesterol metabolism and cardiovascular risk profile in high-risk patients. Ezetimibe is a selective inhibitor of the LDL receptor, which reduces intestinal cholesterol absorption by blocking LDL receptor activity. The combination is also well tolerated and may reduce long-term risk of vascular events such as cardiovascular mortality, non-fatal myocardial infarction, unstable angina, and stroke. Safety and emerging issues Long-term use

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating discovery, improving efficiency, and enabling new types of analysis. With powerful computational tools, scientists can explore larger sample sizes, analyze data at scale, and test hypotheses faster than ever before. At the same time, AI helps interpret results and understand biolog

In [79]:
# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
# creates a folder where to save the merged model.
merged_model_dir = "/content/pharma_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

In [80]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [81]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)

In [82]:
# Merge LoRA adapter weights into the base model weights.(PERMANENTLY)
merged_model = model_with_adapter.merge_and_unload()

In [83]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(merged_model_dir)

inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/pharma_tinyllama_merged_model


## **Stage 2: Continue with Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model**

In Stage 1, we performed **non-instruction fine-tuning / domain-adaptive continued pretraining** on raw pharma PDF text.

Now we continue from the **same Stage 1 LoRA adapter** and perform **instruction fine-tuning** using structured pharma instruction-response examples.
For instruction fine-tuning, the data looks like:

ALPACA FORMAT
```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK..."
}
```



In [84]:
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"

In [85]:
from datasets import load_dataset

In [86]:
# In Hf format
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [87]:
print(instruction_dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic'],
    num_rows: 48
})


In [88]:
print(instruction_dataset[0])

{'instruction': 'Explain the primary mechanism of action of metformin.', 'input': '', 'output': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [89]:
# ============================================================
# Format instruction records
# ============================================================
# We convert every record into Alpaca-style training text.

def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [90]:
instruction_dataset = instruction_dataset.map(format_instruction_record)

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

In [91]:
instruction_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
    num_rows: 48
})

In [92]:
print(instruction_dataset[0]["text"])

### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.


In [93]:
# ============================================================
# Create train-validation split
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)
# renames the "test" key in your dictionary/dataset object to "validation".
instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 8
    })
})
Train examples: 40
Validation examples: 8


In [94]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
# The tokenizer converts text into token IDs for model training.

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


In [95]:
def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Ignore padding tokens in the loss calculation.
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

When we tokenize instruction data, all examples are not the same length.
Example:
```text
Example 1 = 20 tokens
Example 2 = 80 tokens
Example 3 = 150 tokens
But for training, we often make every example of same length equal to max_length = 512
So shorter examples get extra padding tokens.
Example:
Real text tokens + padding tokens = 512 tokens
Now the problem is:
We want the model to learn from real text, not from padding.
So we use -100 in labels.
-100 tells PyTorch:
Ignore this position while calculating loss.
```

In [96]:
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

Tokenizing instruction dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing instruction dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8
    })
})


In [97]:
# # ============================================================
# # Approach 1 code
# # Reload Stage 1 LoRA adapter as trainable
# # ============================================================
# # We continue instruction fine-tuning from the non-instruction LoRA adapter.

# gc.collect()

# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# use_cuda = torch.cuda.is_available()

# if use_cuda:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(
#         config.model_name,
#         quantization_config=BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_quant_type="nf4",
#             bnb_4bit_compute_dtype=torch.float16,
#             bnb_4bit_use_double_quant=True,
#         ),
#         device_map="auto",
#         trust_remote_code=True,
#     )

#     instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

# else:
#     instruction_base_model = AutoModelForCausalLM.from_pretrained(
#         config.model_name,
#         torch_dtype=torch.float32,
#         trust_remote_code=True,
#     )

# instruction_base_model.config.use_cache = False

# # Load the Stage 1 adapter and keep it trainable for Stage 2.
# instruction_model = PeftModel.from_pretrained(
#     instruction_base_model,
#     config.adapter_dir,
#     is_trainable=True,
# )

# instruction_model.print_trainable_parameters()

# Base model
#    +
# Stage 1 domain LoRA adapter
#    ↓ continue training
# Stage 1 + Stage 2 final LoRA adapter

In [98]:
# ============================================================
# Approach 2 code
# Load Merged Stage 1 Model
#    +
# New LoRA adapter for instruction tuning
# ============================================================

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

merged_model_dir = "/content/pharma_tinyllama_merged_model"

if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
instruction_base_model.config.use_cache = False

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [99]:
# ============================================================
# Create a new LoRA adapter for instruction fine-tuning.
# ============================================================

instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [100]:
# ============================================================
# Instruction fine-tuning data collator
# ============================================================
# This prepares mini-batches for causal language model training.

from transformers import DataCollatorForLanguageModeling
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [101]:
# ============================================================
# Instruction fine-tuning arguments
# ============================================================

instruction_output_dir = "/content/pharma_tinyllama_instruction_lora_output"
instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"

os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)

In [103]:
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,

    # Training will stop after 5 optimizer steps.
    # max_steps overrides num_train_epochs.
    num_train_epochs=8,
    max_steps=5,

    # Batch settings.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings.
    learning_rate=1e-4,
    warmup_steps=2,
    weight_decay=0.01,

    # Show training loss at every step.
    logging_steps=1,
    logging_first_step=True,

    # Run validation at every step.
    eval_strategy="steps",
    eval_steps=1,

    # Save checkpoint at final step.
    save_steps=5,
    save_total_limit=2,

    # Precision settings.
    fp16=use_cuda,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,
)

print(instruction_training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

In [104]:
# ============================================================
# Build instruction Trainer
# ============================================================

from transformers import Trainer
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets["train"],
    eval_dataset=instruction_tokenized_datasets["validation"],
    data_collator=instruction_data_collator,
)

print("Instruction Trainer is ready.")

Instruction Trainer is ready.


In [105]:
# ============================================================
# Start instruction fine-tuning
# ============================================================

instruction_train_result = instruction_trainer.train()
print("Instruction fine-tuning completed.")
print(instruction_train_result)


Step,Training Loss,Validation Loss
1,1.585312,1.975368
2,1.902692,1.925296
3,2.093311,1.839250
4,1.708881,1.787779
5,1.728416,1.761966


Instruction fine-tuning completed.
TrainOutput(global_step=5, training_loss=1.8037222385406495, metrics={'train_runtime': 26.8103, 'train_samples_per_second': 1.492, 'train_steps_per_second': 0.186, 'total_flos': 128671096504320.0, 'train_loss': 1.8037222385406495, 'epoch': 1.0})


In [106]:
# ============================================================
# Save final instruction-tuned LoRA adapter
# ============================================================
# This adapter now contains Stage 1 domain adaptation + Stage 2 instruction tuning.

import os

instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"
os.makedirs(instruction_adapter_dir, exist_ok=True)

instruction_trainer.model.save_pretrained(instruction_adapter_dir)
tokenizer.save_pretrained(instruction_adapter_dir)

print(f"Final instruction-tuned LoRA adapter saved to: {instruction_adapter_dir}")
print(os.listdir(instruction_adapter_dir))

Final instruction-tuned LoRA adapter saved to: /content/pharma_tinyllama_instruction_lora_adapter
['adapter_config.json', 'README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [107]:
# ============================================================
# Push Stage 2 instruction LoRA adapter to Hugging Face
# ============================================================

instruction_trainer.model.push_to_hub(
    HF_REPO_INSTRUCTION_ADAPTER,
    private=True
)

tokenizer.push_to_hub(
    HF_REPO_INSTRUCTION_ADAPTER,
    private=True
)

print("Stage 2 instruction LoRA adapter pushed to:")
print(HF_REPO_INSTRUCTION_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 2 instruction LoRA adapter pushed to:
piyushh45/pharma-tinyllama-instruction-lora-adapter


In [108]:
# ============================================================
# Reload final instruction-tuned adapter for inference
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
         merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

final_instruction_model = PeftModel.from_pretrained(
    base_model,
    instruction_adapter_dir,
)

final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Final instruction-tuned model loaded successfully.


In [109]:
# ============================================================
# Instruction-style inference helper
# ============================================================

def build_instruction_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


In [110]:
def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_instruction_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(final_instruction_model.device)

    with torch.no_grad():
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [112]:
# ============================================================
# Test instruction-tuned pharma model
# ============================================================

test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?",
    "Summarize the role of lipid nanoparticles in mRNA vaccines.",
    "Why should AI predictions in drug discovery be experimentally validated?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_instruction_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin acts by inhibiting AMPK, thereby reducing glucose uptake and insulin secretion, as well as reducing hepatic gluconeogenesis. It also reduces hepatic iron accumulation and increases antioxidant activity. Metformin has a relatively narrow therapeutic index, with common side effects including nausea, dizziness, diarrhea, and increased liver enzymes when used overdosage. Common clinical uses include type 2 diabetes mellitus, non-alcoholic steatohepatitis, obesity, and pregnancy risk reduction.

QUESTION:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

### Response:
The liver synthesizes cholesterol, while the blood carries out transportation. When an ApoA1 antibody is low, the liver produces more cholesterol to compensate for the reduced production of ApoA2 antibodies. Cholesterol also helps transport fats through the bloodstream, which reduces inflammation in some people with atherosclerosis. Ezetimibe reduces cholesterol absorption by blocking a lipoprotein receptor-related protein pathway. Atorvastatin increases high-density lipoprotein cholesterol absorption by targeting the HMG-CoA reductase pathway.
QUESTION:
Summarize the role of lipid nanoparticles in mRNA vaccines.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Summarize the role of lipid nanoparticles in mRNA vaccines.

### Response:
Lipid nanoparticles play an important role in mRNA vaccine production. They provide stability and protection against oxidative stress. Lipid nanoparticles also help to stabilize antigen concentration, which is important for efficient immune response. The role of lipids in mRNA vaccines can be summarized as follows:
- Lipid nanoparticles prevent oxidation of antigens.
- Lipid nanoparticles support antigen stability.
- Lipid nanoparticles protect antigens from degradation.


QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:
### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
AI prediction is not an experimental test because it only makes probabilistic claims and does not provide any experimental evidence. The predicted activity is evaluated using a standardized set of measures and can be compared

In [113]:
# ============================================================
# Merge instruction-tuned LoRA adapter into base model
# ============================================================
# This creates a standalone instruction-tuned model.
# Later, we can use this merged model as the base model for preference tuning.

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Path where the final merged instruction-tuned model will be saved.
merged_instruction_model_dir = "/content/pharma_tinyllama_instruction_merged_model"

os.makedirs(merged_instruction_model_dir, exist_ok=True)

In [114]:
# Load the original base model in normal precision for safe merging.
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    merged_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Load the tokenizer.
tokenizer_for_merge = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,
)

if tokenizer_for_merge.pad_token is None:
    tokenizer_for_merge.pad_token = tokenizer_for_merge.eos_token

# Attach the final instruction-tuned LoRA adapter.
model_with_instruction_adapter = PeftModel.from_pretrained(
    base_model_for_merge,
    instruction_adapter_dir,
)

# Merge LoRA adapter weights into the base model weights.
merged_instruction_model = model_with_instruction_adapter.merge_and_unload()

# Save the standalone merged model and tokenizer.
merged_instruction_model.save_pretrained(merged_instruction_model_dir)
tokenizer_for_merge.save_pretrained(merged_instruction_model_dir)

print(f"Merged instruction-tuned model saved to: {merged_instruction_model_dir}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged instruction-tuned model saved to: /content/pharma_tinyllama_instruction_merged_model


## **Stage 3: Preference Tuning with DPO**

In Stage 1, we adapted the model to the pharma domain using raw non-instruction text.

In Stage 2, we instruction-tuned the model using instruction-response data.

In Stage 3, we will use **preference data** with DPO.

DPO data has three main columns:

```text
prompt
chosen
rejected
```


In [115]:
# ============================================================
# 25. Install TRL for DPO training
# ============================================================
# TRL provides DPOTrainer and DPOConfig for preference tuning.

!pip install -q -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 23.8 MB/s eta 0:00:00


In [116]:
# ============================================================
# 26. Load DPO preference dataset
# ============================================================
# Expected columns: prompt, chosen, rejected

from datasets import load_dataset

preference_data_path = "/content/pharma_preference_dataset.jsonl"

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train"
)

print(preference_dataset)
print(preference_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
    num_rows: 48
})
{'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:\n', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [117]:
# Create train-validation split
preference_dataset = preference_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

# Rename test split to validation split
preference_dataset["validation"] = preference_dataset.pop("test")

print("After train-validation split:")
print(preference_dataset)
print("Train rows:", len(preference_dataset["train"]))
print("Validation rows:", len(preference_dataset["validation"]))

After train-validation split:
DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 8
    })
})
Train rows: 40
Validation rows: 8


In [118]:
# ============================================================
# Load merged instruction model as base for preference tuning
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

use_cuda = torch.cuda.is_available()

if use_cuda:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    preference_base_model = prepare_model_for_kbit_training(preference_base_model)

else:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_base_model.config.use_cache = False

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [119]:
# ============================================================
# Create a new LoRA adapter for preference tuning.
# ============================================================

preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

preference_model = get_peft_model(
    preference_base_model,
    preference_lora_config,
)

preference_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [120]:
# ============================================================
# 30. Configure DPO training
# ============================================================

import os
from trl import DPOTrainer
from trl import DPOConfig

try:
    from trl import DPOConfig
    has_dpo_config = True
except ImportError:
    DPOConfig = None
    has_dpo_config = False

preference_output_dir = "/content/pharma_tinyllama_preference_dpo_output"
preference_adapter_dir = "/content/pharma_tinyllama_preference_dpo_lora_adapter"

os.makedirs(preference_output_dir, exist_ok=True)
os.makedirs(preference_adapter_dir, exist_ok=True)

In [121]:
# dpo_eval_mode = "steps" if len(preference_dataset["validation"]) > 0 else "no"

# # DPO-specific settings.
# # beta controls how strongly the model is pushed toward chosen answers over rejected answers.
# dpo_config_kwargs = dict(
#     output_dir=preference_output_dir,
#     num_train_epochs=3,
#     max_steps=5,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=5e-5,
#     warmup_steps=5,
#     weight_decay=0.01,
#     logging_steps=1,
#     logging_first_step=True,
#     eval_steps=1,
#     save_steps=5,
#     save_total_limit=2,
#     fp16=False,
#     bf16=False,
#     report_to="none",
#     remove_unused_columns=False,

#     # DPO hyperparameters.
#     beta=0.1,
#     max_length=512,
#     max_prompt_length=256,
# )

# if has_dpo_config:
#     # Newer TRL versions use DPOConfig.
#     dpo_config_params = inspect.signature(DPOConfig.__init__).parameters

#     if "eval_strategy" in dpo_config_params:
#         dpo_config_kwargs["eval_strategy"] = dpo_eval_mode
#     elif "evaluation_strategy" in dpo_config_params:
#         dpo_config_kwargs["evaluation_strategy"] = dpo_eval_mode

#     safe_dpo_config_kwargs = {
#         key: value
#         for key, value in dpo_config_kwargs.items()
#         if key in dpo_config_params
#     }

#     removed_dpo_args = set(dpo_config_kwargs.keys()) - set(safe_dpo_config_kwargs.keys())
#     if removed_dpo_args:
#         print("Removed unsupported DPOConfig arguments:", removed_dpo_args)

#     dpo_training_args = DPOConfig(**safe_dpo_config_kwargs)

# else:
#     # Older TRL versions may use TrainingArguments.
#     training_args_params = inspect.signature(TrainingArguments.__init__).parameters

#     training_kwargs = {
#         key: value
#         for key, value in dpo_config_kwargs.items()
#         if key in training_args_params
#     }

#     if "eval_strategy" in training_args_params:
#         training_kwargs["eval_strategy"] = dpo_eval_mode
#     elif "evaluation_strategy" in training_args_params:
#         training_kwargs["evaluation_strategy"] = dpo_eval_mode

#     dpo_training_args = TrainingArguments(**training_kwargs)

# print(dpo_training_args)


In [122]:
# # ============================================================
# # 31. Build DPOTrainer
# # ============================================================

# import inspect
# from trl import DPOTrainer

# dpo_trainer_kwargs = dict(
#     model=preference_model,
#     ref_model=None,
#     args=dpo_training_args,
#     train_dataset=preference_dataset["train"],
#     eval_dataset=preference_dataset["validation"] if len(preference_dataset["validation"]) > 0 else None,
# )

# dpo_trainer_params = inspect.signature(DPOTrainer.__init__).parameters

# # TRL versions differ: newer versions use processing_class, older versions use tokenizer.
# if "processing_class" in dpo_trainer_params:
#     dpo_trainer_kwargs["processing_class"] = tokenizer
# elif "tokenizer" in dpo_trainer_params:
#     dpo_trainer_kwargs["tokenizer"] = tokenizer

# # Older TRL versions may expect these DPO parameters directly in the trainer.
# if "beta" in dpo_trainer_params:
#     dpo_trainer_kwargs["beta"] = 0.1
# if "max_length" in dpo_trainer_params:
#     dpo_trainer_kwargs["max_length"] = 512
# if "max_prompt_length" in dpo_trainer_params:
#     dpo_trainer_kwargs["max_prompt_length"] = 256

# safe_dpo_trainer_kwargs = {
#     key: value
#     for key, value in dpo_trainer_kwargs.items()
#     if key in dpo_trainer_params
# }

# removed_trainer_args = set(dpo_trainer_kwargs.keys()) - set(safe_dpo_trainer_kwargs.keys())
# if removed_trainer_args:
#     print("Removed unsupported DPOTrainer arguments:", removed_trainer_args)

# dpo_trainer = DPOTrainer(**safe_dpo_trainer_kwargs)

# print("DPOTrainer is ready.")

In [123]:
# ============================================================
# 30. Create DPO training arguments - Simple Version
# ============================================================

from trl import DPOConfig

dpo_training_args = DPOConfig(
    output_dir=preference_output_dir,

    # Training duration
    num_train_epochs=6,
    max_steps=5,

    # Batch settings
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings
    learning_rate=5e-5,
    warmup_steps=2,
    weight_decay=0.01,

    # Logging and evaluation
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=1,

    # Checkpoint saving
    save_steps=5,
    save_total_limit=2,

    # Precision settings
    fp16=False,
    bf16=False,

    # Disable external logging tools
    report_to="none",

    # Keep required columns
    remove_unused_columns=False,

    # DPO hyperparameter
    beta=0.1,
)

print(dpo_training_args)

DPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.1,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoin

In [124]:
# ============================================================
# 31. Build DPOTrainer
# ============================================================

from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,  # None means TRL will internally use the reference behavior
    args=dpo_training_args,

    train_dataset=preference_dataset["train"],
    eval_dataset=preference_dataset["validation"],

    processing_class=tokenizer,
)

print("DPOTrainer is ready.")

Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DPOTrainer is ready.


In [125]:
# ============================================================
# 32. Start DPO preference tuning
# ============================================================

dpo_train_result = dpo_trainer.train()

print("DPO preference tuning completed.")
print(dpo_train_result)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.693147,0.693147,1.811863,1413.000000,-3.054256,-3.236887,0.644058,0.000000,0.000000,0.000000,0.000000,-103.842898,-123.766562
2,0.693147,0.631585,1.805925,2789.000000,-3.048932,-3.235095,0.642434,0.010489,-0.118531,1.000000,0.129020,-103.738012,-124.951874
3,0.641278,0.475783,1.794639,4051.000000,-3.038281,-3.230597,0.642434,0.032540,-0.488301,1.000000,0.520841,-103.517499,-128.649576
4,0.493199,0.394911,1.787527,5380.000000,-3.031501,-3.226959,0.644058,0.046165,-0.732951,1.000000,0.779117,-103.381247,-131.096076
5,0.395025,0.360362,1.784468,6698.000000,-3.028058,-3.225296,0.647932,0.052435,-0.849680,1.000000,0.902114,-103.318550,-132.263358


DPO preference tuning completed.
TrainOutput(global_step=5, training_loss=0.5831593692302703, metrics={'train_runtime': 62.5145, 'train_samples_per_second': 0.64, 'train_steps_per_second': 0.08, 'total_flos': 49278084096000.0, 'train_loss': 0.5831593692302703, 'epoch': 1.0})


| Parameter               | Short Meaning                                                                              |
| ----------------------- | ------------------------------------------------------------------------------------------ |
| **Step**                | Current optimizer step during training.                                                    |
| **Training Loss**       | DPO loss on the training data; lower is generally better.                                  |
| **Validation Loss**     | DPO loss on unseen validation data; helps check generalization.                            |
| **Entropy**             | Measures how uncertain the model is; higher means more random, lower means more confident. |
| **Num Tokens**          | Total number of tokens processed so far.                                                   |
| **Logits/chosen**       | Raw model score for the preferred answer.                                                  |
| **Logits/rejected**     | Raw model score for the rejected answer.                                                   |
| **Mean Token Accuracy** | Average token-level prediction accuracy.                                                   |
| **Rewards/chosen**      | DPO implicit reward for the preferred answer; should be higher.                            |
| **Rewards/rejected**    | DPO implicit reward for the rejected answer; should be lower.                              |
| **Rewards/accuracies**  | How often the model ranks the chosen answer above the rejected answer.                     |
| **Rewards/margins**     | Difference between chosen reward and rejected reward; positive is good.                    |
| **Logps/chosen**        | Log probability of the chosen answer; less negative means more likely.                     |
| **Logps/rejected**      | Log probability of the rejected answer; ideally more negative than chosen.                 |


Simple summary: In DPO training, the main goal is to make the model assign higher probability and higher reward to the chosen answer than the rejected answer.

In [126]:
# ============================================================
# 33. Save DPO preference-tuned LoRA adapter
# ============================================================

dpo_trainer.model.save_pretrained(preference_adapter_dir)
tokenizer.save_pretrained(preference_adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {preference_adapter_dir}")
print(os.listdir(preference_adapter_dir))


Preference-tuned LoRA adapter saved to: /content/pharma_tinyllama_preference_dpo_lora_adapter
['adapter_config.json', 'README.md', 'adapter_model.safetensors', 'ref', 'tokenizer_config.json', 'tokenizer.json']


In [127]:
# ============================================================
# Push Stage 3 DPO LoRA adapter to Hugging Face
# ============================================================

dpo_trainer.model.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=True
)

tokenizer.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=True
)

print("Stage 3 DPO LoRA adapter pushed to:")
print(HF_REPO_DPO_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

  ...adapter_model.safetensors:   1%|1         |  351kB / 25.3MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 3 DPO LoRA adapter pushed to:
piyushh45/pharma-tinyllama-dpo-lora-adapter


In [128]:
# ============================================================
# 34. Reload preference-tuned model for inference
# ============================================================

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_inference_model = PeftModel.from_pretrained(
    preference_inference_base_model,
    preference_adapter_dir,
)

preference_inference_model.eval()

print("Preference-tuned model loaded successfully for inference.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Preference-tuned model loaded successfully for inference.


In [129]:
# ============================================================
# 35. Preference-tuned inference helper
# ============================================================

def build_preference_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [130]:
def generate_preference_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_preference_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(preference_inference_model.device)

    with torch.no_grad():
        outputs = preference_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [131]:
# ============================================================
# 36. Test preference-tuned pharma model
# ============================================================

preference_test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why should AI predictions in drug discovery be experimentally validated?",
    "Define pharmacovigilance.",
    "Explain why pharmacovigilance continues after drug approval.",
]

for question in preference_test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_preference_response(question, max_new_tokens=150))


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin reduces glucose in the liver, which causes sensitivity to insulin and activation of AMP-activated protein kinase (AMPK). This induces an antioxidant state through downregulation of AMPKα1 and upregulation of AMPKα2, leading to reduced oxidative stress. It also activates mTOR, which regulates cell growth and survival through mechanisms such as reduction of p-Akt and phosphorylation of mTORC1 substrates. Metformin acts primarily through its ability to reduce intracellular glucose levels, but it may also act through downstream signaling mechanisms that increase cellular
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
AI prediction is only as good as the data used to train it. In drug discovery, biologically active compounds are generated using chemical synthesis and validation steps. Validation can include chemical properties, assay sensitivity, cellular toxicity, and molecular property prediction. If an AI model produces a predicted compound that does not meet experimental requirements, it may be difficult or even impossible to evaluate its activity in vivo. Therefore, experimental validation is essential for AI prediction accuracy in drug discovery. 

### Instruction:
What is drug-likeness prediction? Why might it be important? 

### Response:
Drug-likeness prediction aims to predict whether a
QUESTION:
Define pharmacovigilance.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Define pharmacovigilance.

### Response:
Pharmacovigilance is the collection and analysis of information about medicines to monitor safety, quality, efficacy, distribution, use, and regulatory compliance. It is a core part of drug development, with primary focus on adverse event reporting and clinical research. Pharmacovigilance also plays an important role in post-market surveillance, where unexpected events can be identified and addressed early, reducing risk for patients. 

### Question:
What are the main features of pharmacovigilance?

### Instruction:
Define patient engagement.

### Response:
Patient engagement refers to active involvement by patients in health care
QUESTION:
Explain why pharmacovigilance continues after drug approval.

MODEL RESPONSE:
### Instruction:
Explain why pharmacovigilance continues after drug approval.

### Response:
Pharmacovigilance is important after drug approval because it can help prevent adverse events, detect early signs of safet

In [132]:
# ============================================================
# 37. Optional: Merge DPO preference adapter into the instruction-tuned base model
# ============================================================
# Use this only after preference tuning is complete and you want a standalone final model.

import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_merged_preference_model_dir = "/content/pharma_tinyllama_final_preference_merged_model"
os.makedirs(final_merged_preference_model_dir, exist_ok=True)

In [133]:
# Load the merged instruction model in normal precision for safe merging.
base_model_for_preference_merge = AutoModelForCausalLM.from_pretrained(
    merged_instruction_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Attach the DPO preference LoRA adapter.
model_with_preference_adapter = PeftModel.from_pretrained(
    base_model_for_preference_merge,
    preference_adapter_dir,
)

# Merge the preference adapter into the instruction-tuned base model.
final_merged_preference_model = model_with_preference_adapter.merge_and_unload()

# Save final standalone model and tokenizer.
final_merged_preference_model.save_pretrained(final_merged_preference_model_dir)
tokenizer.save_pretrained(final_merged_preference_model_dir)

print(f"Final merged preference-tuned model saved to: {final_merged_preference_model_dir}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final merged preference-tuned model saved to: /content/pharma_tinyllama_final_preference_merged_model
